# Higgins et al. (2017), *β-VAE* — Figure 7A/7B on dSprites

A VAE ($\beta=1$) and a $\beta$-VAE ($\beta=4$) with the paper's fully connected architecture are trained on the 737,280
dSprites images with Adagrad (lr 0.01). `analyze.py` then draws, per latent (ordered by learnt variance): the mean
latent response to position / scale / rotation, and the decoder output when that latent is traversed over $[-3,3]$.

Code: `data.py`, `model.py`, `train.py`, `analyze.py`; `run_all.sh` runs everything (about 1 h on an Apple M4 Pro).

In [ ]:
import os, json, subprocess
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
RUN_TRAINING = False
if RUN_TRAINING:
    subprocess.run(["bash", "run_all.sh"], check=True)

## Training curves: reconstruction term and KL term

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for b, c in [(1, "tab:blue"), (4, "tab:red")]:
    f = f"results/log_beta{b}_s0.json"
    if not os.path.exists(f): continue
    d = json.load(open(f))
    axes[0].plot(d["epoch"], d["recon"], color=c, label=f"beta={b}"); axes[1].plot(d["epoch"], d["kl"], color=c, label=f"beta={b}")
    print(f"beta={b}: final recon {d['recon'][-1]:.2f} nats, KL {d['kl'][-1]:.2f} nats, KL per latent {np.round(d['kl_per_dim'][-1], 2).tolist()}")
axes[0].set_ylabel("-E[log p(x|z)] (nats)"); axes[1].set_ylabel("KL(q(z|x)||p(z)) (nats)")
for ax in axes: ax.set_xlabel("epoch"); ax.legend(); ax.grid(ls=":")
plt.tight_layout(); plt.show()

## Figure 7A/7B reproduced

In [ ]:
!python3 analyze.py --betas 4,1 --out figures/fig7ab_reproduced.png
display(Image("figures/fig7ab_reproduced.png"))

## Reconstructions: the price of a larger β

In [ ]:
import torch
from data import load_dsprites
from model import BetaVAE
from train import get_device
dev = get_device(); imgs, classes = load_dsprites(os.environ.get("DSPRITES_NPZ"))
idx = np.random.RandomState(0).choice(len(imgs), 8, replace=False)
x = torch.from_numpy(imgs[idx]).to(dev).float()
fig, axes = plt.subplots(3, 8, figsize=(12, 4.6))
for j in range(8): axes[0, j].imshow(imgs[idx[j]], cmap="gray"); axes[0, j].axis("off")
axes[0, 0].set_title("data", loc="left")
for r, b in enumerate([1, 4]):
    m = BetaVAE().to(dev); m.load_state_dict(torch.load(f"results/vae_beta{b}_s0.pt", map_location=dev)); m.eval()
    with torch.no_grad():
        mu, _ = m.encode(x); rec = torch.sigmoid(m.decode(mu)).view(-1, 64, 64).cpu().numpy()
    for j in range(8): axes[r+1, j].imshow(rec[j], cmap="gray", vmin=0, vmax=1); axes[r+1, j].axis("off")
    axes[r+1, 0].set_title(f"reconstruction, beta={b}", loc="left")
plt.tight_layout(); plt.show()